# GAT Native Adapter v1

目标：把 `gat_mse` 从 attention feature Ridge proxy 推进到 `src/model` 的 native graph attention adapter。Notebook 只保留实验流程、外部参考 URL 和中文结论；可复用训练、回测、analysis 逻辑都调用 `src/`。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'AGENTS.md').exists() and (path / 'src').exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import build_supervised_dataset, load_soxl_support_frame
from src.model.capabilities import model_capability_table, model_fidelity_summary
from src.model.graph import GATGraphRegressor
from src.model.matrix import run_quantbench_matrix_smoke
from src.model.training import fit_estimator_for_model, model_adapter_name

OUTPUT_DIR = REPO_ROOT / 'research' / 'gat_native_adapter_v1'
CARD_DIR = OUTPUT_DIR / 'matrix_card'

outputs = run_quantbench_matrix_smoke(
    card_dir=CARD_DIR,
    start='2026-01-01',
    end='2026-06-09',
    train_end='2026-04-30',
    model_names=('ridge_mse', 'gcn_mse', 'gat_mse', 'rgcn_mse'),
)

frame = load_soxl_support_frame(start='2026-01-01', end='2026-06-09')
dataset = build_supervised_dataset(frame, horizon=1)
train = dataset.frame.loc[dataset.frame['date'] <= '2026-04-30'].copy()
feature_columns = tuple(dataset.feature_columns[:8])
estimator, constant_prediction = fit_estimator_for_model('gat_mse', train=train, feature_columns=feature_columns)
capability = model_capability_table(('gat_mse',)).iloc[0].to_dict()
fidelity = model_fidelity_summary()

assert constant_prediction is None
assert isinstance(estimator, GATGraphRegressor)
assert estimator.feature_count_ == 8
assert estimator.node_count_ == 8
assert estimator.attention_weights_.shape == (8, 8)
assert hasattr(estimator, 'model_')
assert capability['paper_fidelity'] == 'native_graph'
assert capability['is_proxy'] is False
assert model_adapter_name('gat_mse') == 'gat_graph_attention_mse_v1'
assert 'gat_mse' in fidelity['native_models']
assert 'gat_mse' not in fidelity['proxy_models']
assert len(fidelity['native_models']) == 22
assert len(fidelity['proxy_models']) == 10

summary = {
    'schema_version': 'research.gat_native_adapter_v1',
    'status': 'completed',
    'model': 'gat_mse',
    'adapter': model_adapter_name('gat_mse'),
    'estimator': estimator.__class__.__name__,
    'node_count': estimator.node_count_,
    'feature_count': estimator.feature_count_,
    'hidden_size': estimator.hidden_size,
    'paper_fidelity': capability['paper_fidelity'],
    'is_proxy': bool(capability['is_proxy']),
    'full_matrix_native_model_count': len(fidelity['native_models']),
    'full_matrix_proxy_model_count': len(fidelity['proxy_models']),
    'matrix_card': str(CARD_DIR.relative_to(REPO_ROOT)),
    'paper_reference': {
        'source_id': 'arxiv:1710.10903',
        'url': 'https://arxiv.org/abs/1710.10903',
        'usage': '自有摘要和结构假设；实现只复用 masked self-attention over graph neighborhoods 的思想，不复制论文全文或第三方代码。',
    },
    'artifact_links': {
        'analysis_summary': str((CARD_DIR / 'analysis_summary.json').relative_to(REPO_ROOT)),
        'model_comparison': str((CARD_DIR / 'model_comparison.csv').relative_to(REPO_ROOT)),
        'model_run_ledger': str((CARD_DIR / 'model_run_ledger.csv').relative_to(REPO_ROOT)),
        'manifest': str((CARD_DIR / 'analysis_manifest.json').relative_to(REPO_ROOT)),
    },
}
(OUTPUT_DIR / 'notebook_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
summary


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'schema_version': 'research.gat_native_adapter_v1',
 'status': 'completed',
 'model': 'gat_mse',
 'adapter': 'gat_graph_attention_mse_v1',
 'estimator': 'GATGraphRegressor',
 'node_count': 8,
 'feature_count': 8,
 'hidden_size': 8,
 'paper_fidelity': 'native_graph',
 'is_proxy': False,
 'full_matrix_native_model_count': 22,
 'full_matrix_proxy_model_count': 10,
 'matrix_card': 'research/gat_native_adapter_v1/matrix_card',
 'paper_reference': {'source_id': 'arxiv:1710.10903',
  'url': 'https://arxiv.org/abs/1710.10903',
  'usage': '自有摘要和结构假设；实现只复用 masked self-attention over graph neighborhoods 的思想，不复制论文全文或第三方代码。'},
 'artifact_links': {'analysis_summary': 'research/gat_native_adapter_v1/matrix_card/analysis_summary.json',
  'model_comparison': 'research/gat_native_adapter_v1/matrix_card/model_comparison.csv',
  'model_run_ledger': 'research/gat_native_adapter_v1/matrix_card/model_run_ledger.csv',
  'manifest': 'research/gat_native_adapter_v1/matrix_card/analysis_manifest.json'}}

## 外部参考摘要

- Velickovic et al., `Graph Attention Networks`：核心结构是对图邻域中的节点表征做 masked self-attention，用学习到的注意力权重替代固定聚合权重。参考：https://arxiv.org/abs/1710.10903
- 本仓库实现：SOXL 日频监督任务当前没有稳定的真实资产关系图，本轮先把特征列视为 feature nodes，并用训练窗口相关性作为 attention prior。后续若引入 SOXX/VIX/QQQ/SPY 或行业成分真实关系，应替换 graph provider，而不是把关系硬编码在 notebook。

## 中文结论

- 做了什么：在 `src/model.graph` 增加 `GATGraphRegressor`，把 `gat_mse` 从 `GraphFeatureRidgeRegressor(graph_kind='gat')` proxy 升级为 `native_graph`。实现使用训练窗口特征相关性作为 attention prior，并在 feature-node 图上学习 graph attention pooling 后回归预测。
- 核心指标：本 notebook 训练并回测 `ridge_mse`、`gcn_mse`、`gat_mse`、`rgcn_mse`，输出 `model_comparison.csv`、`daily_backtest.csv`、`model_run_ledger.csv`、`qlib_risk_metrics.csv`、`model_effect_deltas.csv`、`model_regime_window_diagnostics.csv` 和 manifest。
- 模型实现状态：全模型注册表当前 native=22、proxy=10；`gat_mse` 与 `gcn_mse` 均进入 `native_graph`。
- Regime/区间差异：本轮只证明 GAT native adapter 能进入统一回测和 analysis contract；具体 regime winner/loser 以 `matrix_card/analysis_summary.json` 和 `model_review_briefing.csv` 为准。
- 是否晋升：否。本次是模型实现和链路覆盖升级，不直接晋升生产策略。
- 风险：当前 feature-node graph 是相关性先验，不是真实 support universe 关系图；训练窗口较短，attention 权重需要更长窗口和更真实图结构复核。
- 下一步：刷新全模型 ledger，然后继续替换 `rgcn_mse`、`rsr_mse`、`hypergraph_mse` 等剩余 graph proxy。